**[Course Home](../README.md) | [Python Lab](../python/05_service_variance_and_pollaczek_khinchine.ipynb) | [Prev: Ch 04: Renewal Theory & Inspection Paradox](04_the_inspection_paradox_and_renewal_theory.ipynb) | [Next: Ch 06: Kingman Heavy-Traffic](../README.md)**

*Part II: Renewal Theory, Variance & Heavy Traffic — Chapter 05 of 11*

---

# Chapter 5: The Variance Penalty — $M/G/1$ Queues, Pollaczek-Khinchine Formula, and How Heavy-Tailed Jobs Stall Linearly Stable Systems
### *The Pollaczek-Khinchine Relation, The $C_v^2$ Amplification Factor, The Infinite Variance Catastrophe, and The ROI of Test Suite Sharding*

---

## 1. The Core Engineering Problem: The Hidden Destroyer of Latency

In performance tuning and fleet capacity planning, engineers routinely obsess over **utilization**:

> *"Our worker fleet is running at 70% average utilization. Since utilization is well below 100%, queue delays should be modest and manageable."*

Yet in production, systems running at 60% or 70% utilization frequently experience catastrophic queue backlogs, with jobs idling in buffers for hours.

When engineers investigate the culprit, they discover that the root cause is not traffic volume; it is **service time variability**.

A queueing system does not merely suffer from high traffic intensity $\rho$. It suffers an equal or greater penalty from the **dispersion of job execution times**. When a 2-hour integration benchmark is mixed into the same FIFO queue as 10-second unit tests, the slow job acts like a massive boulder lodged in a pipeline.

Behind that boulder, dozens of lightweight, fast-completing jobs accumulate in a growing convoy.

```
Incoming Stream:    [10s] [10s] [10s] [10s] [10s] [10s] [10s] [   2-Hour Integration Benchmark   ]
                    \-------------------------------------/   |
                                       |                      v
                             Convoy of Fast Jobs         Server Busy for 2 Hours!
                             Blocked in FIFO Buffer      (Blocks all downstream flow)
```

The mathematical equation that governs this reality is the **Pollaczek-Khinchine (P-K) Formula**. It reveals that queue waiting time is the product of two independent factors:
1. **The Utilization Factor**: $\frac{\rho}{1 - \rho}$ (the congestion hockey stick).
2. **The Variance Factor**: $\frac{1 + C_v^2}{2}$ (the variability penalty).

If variance is high, a system can be lightly loaded ($\rho = 0.40$) and yet exhibit longer queue delays than a heavily loaded system ($\rho = 0.85$) with uniform, predictable runtimes!

---

## 2. The $M/G/1$ Queue Architecture

We generalize from the memoryless $M/M/1$ queue to the **$M/G/1$ queue**:
* **$M$ (Markovian Arrivals)**: Jobs arrive as a homogeneous Poisson process with rate $\lambda$. Inter-arrival times are i.i.d. $\text{Exp}(\lambda)$.
* **$G$ (General Service Times)**: Job service durations $S$ are drawn independently from an arbitrary probability distribution $G$ with:
  * Mean service duration: $\mathbb{E}[S] = \frac{1}{\mu}$.
  * Service time variance: $\sigma_S^2 = \text{Var}(S) = \mathbb{E}[S^2] - (\mathbb{E}[S])^2$.
  * Squared coefficient of variation:
    $$
    C_v^2 = \frac{\sigma_S^2}{(\mathbb{E}[S])^2}
    $$
* **$1$ (Single Server)**: A single dedicated worker processes jobs one at a time.
* **FIFO Discipline & Infinite Buffer**: Jobs are served in the order they arrive.

The traffic intensity (utilization) is:

$$
\rho = \lambda \mathbb{E}[S] = \frac{\lambda}{\mu}
$$

The stability condition is $\rho < 1.0$.

---

## 3. Derivation of the Pollaczek-Khinchine (P-K) Formula

The derivation of the P-K formula is one of the most elegant applications of **Renewal Theory** and **PASTA** in applied mathematics.

Consider a newly arriving job, which we label Job $A$. Job $A$ arrives at steady state. 
Its total waiting time in queue $W_q$ is composed of two distinct components:

1. **Residual Service of the Active Job**: If the server is busy when Job $A$ arrives, Job $A$ must wait for the current job to complete. This is the **residual life** $R$ of the job in service.
2. **Execution of Preceding Queued Jobs**: Job $A$ must wait for all $L_q$ jobs that arrived before it and are already waiting in the queue buffer. Each of those jobs will take an independent service duration with mean $\mathbb{E}[S]$.

Taking the mathematical expectation:

$$
\mathbb{E}[W_q] = \mathbb{P}(\text{Server Busy}) \cdot \mathbb{E}[R] + \mathbb{E}[L_q] \cdot \mathbb{E}[S]
$$

```
Timeline of Arriving Job A:
+----------------------------------------------------+--------------------------+
| Residual Life R of Active Job | Job 1 | Job 2 | ...| Service of Job A (S_A)  |
+----------------------------------------------------+--------------------------+
  \-------------------------------------------------/ \------------------------/
              Queue Waiting Time W_q                     Execution Time S_A
```

Let us evaluate each term:
1. **Probability Server is Busy**: By the PASTA property (Chapter 02), Poisson arrivals see time averages. The fraction of time the server is busy is identically the server utilization $\rho$:
   $$\mathbb{P}(\text{Server Busy}) = \rho = \lambda \mathbb{E}[S]$$

2. **Expected Residual Life $\mathbb{E}[R]$**: From Chapter 04 (Renewal Theory & The Inspection Paradox), the expected remaining duration of a renewal interval sampled at random is:
   $$\mathbb{E}[R] = \frac{\mathbb{E}[S^2]}{2 \mathbb{E}[S]}$$

3. **Expected Preceding Backlog $\mathbb{E}[L_q]$**: By Little's Law (Chapter 01) applied to the queue subsystem:
   $$\mathbb{E}[L_q] = \lambda \mathbb{E}[W_q]$$

Now substitute these three identities back into the waiting time balance equation:

$$
W_q = \rho \cdot \left(\frac{\mathbb{E}[S^2]}{2 \mathbb{E}[S]}\right) + (\lambda W_q) \cdot \mathbb{E}[S]
$$

Since $\rho = \lambda \mathbb{E}[S]$, the first term simplifies:

$$
\rho \cdot \left(\frac{\mathbb{E}[S^2]}{2 \mathbb{E}[S]}\right) = (\lambda \mathbb{E}[S]) \cdot \left(\frac{\mathbb{E}[S^2]}{2 \mathbb{E}[S]}\right) = \frac{\lambda \mathbb{E}[S^2]}{2}
$$

Substitute this back:

$$
W_q = \frac{\lambda \mathbb{E}[S^2]}{2} + \rho W_q
$$

Subtract $\rho W_q$ from both sides:

$$
W_q(1 - \rho) = \frac{\lambda \mathbb{E}[S^2]}{2}
$$

Dividing by $(1 - \rho)$ yields the **Pollaczek-Khinchine Mean Waiting Time Formula**:

$$
W_q = \frac{\lambda \mathbb{E}[S^2]}{2(1 - \rho)}
$$

---

## 4. Deconstructing the Formula: The Twin Multipliers

We can rewrite $\mathbb{E}[S^2]$ in terms of the mean $\mathbb{E}[S]$ and the squared coefficient of variation $C_v^2$:

$$
\sigma_S^2 = \mathbb{E}[S^2] - (\mathbb{E}[S])^2 \implies \mathbb{E}[S^2] = (\mathbb{E}[S])^2 + \sigma_S^2 = (\mathbb{E}[S])^2 (1 + C_v^2)
$$

Substitute this into the P-K equation:

$$
W_q = \frac{\lambda (\mathbb{E}[S])^2 (1 + C_v^2)}{2(1 - \rho)} = \left(\frac{\lambda \mathbb{E}[S]}{1 - \rho} \cdot \mathbb{E}[S]\right) \cdot \left(\frac{1 + C_v^2}{2}\right)
$$

Since $\lambda \mathbb{E}[S] = \rho$:

$$
W_q = \underbrace{\left(\frac{\rho}{1 - \rho} \cdot \mathbb{E}[S]\right)}_{\text{Utilization Factor}} \times \underbrace{\left(\frac{1 + C_v^2}{2}\right)}_{\text{Variance Multiplier}}
$$

Applying Little's Law ($L_q = \lambda W_q$), the average queue backlog is:

$$
L_q = \underbrace{\left(\frac{\rho^2}{1 - \rho}\right)}_{\text{M/M/1 Backlog}} \times \underbrace{\left(\frac{1 + C_v^2}{2}\right)}_{\text{Variance Multiplier}}
$$

And the total turnaround time is:

$$
W = W_q + \mathbb{E}[S]
$$

### Analysis of the Variance Multiplier $\frac{1 + C_v^2}{2}$

| Service Distribution | Characteristic | $C_v^2$ | Variance Multiplier $\frac{1 + C_v^2}{2}$ | Mean Queue Wait $W_q$ |
| :--- | :--- | :---: | :---: | :---: |
| **Deterministic ($M/D/1$)** | Zero variance (fixed duration) | **$0.0$** | **$0.50$** | $\frac{1}{2} W_{q, M/M/1}$ |
| **Erlang-$k$ ($M/E_k/1$)** | Sum of $k$ exponential stages | **$1/k$** | $\frac{k+1}{2k}$ | Between $0.5$ and $1.0 \times M/M/1$ |
| **Exponential ($M/M/1$)** | Memoryless baseline | **$1.0$** | **$1.00$** | Standard $M/M/1$ baseline |
| **Hyperexponential ($M/H_2/1$)** | Bimodal mixture (fast & slow) | **$4.0$** | **$2.50$** | $2.5\times$ slower than $M/M/1$ |
| **Log-Normal** | Long positive skew | **$9.0$** | **$5.00$** | $5.0\times$ slower than $M/M/1$ |
| **Pareto (Power-Law)** | Heavy-tailed ($\alpha \le 2$) | **$\infty$** | **$\infty$** | **$\infty$ (Infinite Waiting Time!)** |

### Critical Takeaways:
1. **The Deterministic Half-Price Discount**: Simply by making job execution times perfectly uniform ($C_v^2 \to 0$), queue waiting time is **cut in half** without adding a single extra server!
2. **The Variance Explosion**: If job runtimes are highly variable ($C_v^2 = 9$), waiting time is **five times worse** than predicted by standard $M/M/1$ theory.

---

## 5. The Catastrophe of Infinite Variance: Heavy Tails & Stability Illusion

Consider a system where job execution times follow a Pareto distribution with shape parameter $\alpha \in (1, 2]$:

$$
F(s) = 1 - \left(\frac{s_m}{s}\right)^\alpha, \quad s \ge s_m > 0
$$

For $\alpha > 1$, the first moment is finite:

$$
\mathbb{E}[S] = \frac{\alpha s_m}{\alpha - 1} < \infty
$$

Therefore, we can easily choose an arrival rate $\lambda$ such that:

$$
\rho = \lambda \mathbb{E}[S] < 1.0
$$

The system is **linearly stable**. On average, work arrives slower than the server clears it.

However, for $\alpha \le 2$, the second moment diverges:

$$
\mathbb{E}[S^2] = \int_{s_m}^\infty s^2 f(s) \, ds = \infty \implies \sigma_S^2 = \infty \implies C_v^2 = \infty
$$

Now examine the Pollaczek-Khinchine formula:

$$
W_q = \frac{\lambda \mathbb{E}[S^2]}{2(1 - \rho)} = \frac{\lambda \cdot \infty}{2(1 - \rho)} = \mathbf{\infty}
$$

### The Operational Paradox:
Even if utilization is only **$\rho = 0.20$ (the server is idle 80% of the time)**, the average waiting time experienced by jobs is **INFINITE**!

In practice, this means:
* The queue length exhibits wild, non-stationary fluctuations.
* Standard sample averages of latency do not converge; they drift upward as the simulation runs longer.
* Rare "black swan" monster jobs paralyze the queue, creating backlogs that take hours or days to clear.

---

## 6. Concrete Engineering Walkthrough: Test Suite Sharding vs Hardware Purchasing

Let us compare two engineering proposals to solve severe CI merge queue latency.

### The Baseline System:
* A continuous integration test runner fleet processes PRs.
* Pull requests arrive at rate $\lambda = 12$ PRs/hour ($0.20$ PRs/min).
* Mean test duration: $\mathbb{E}[S] = 4.0$ minutes ($\mu = 0.25$ PRs/min).
* Fleet utilization: $\rho = \lambda \mathbb{E}[S] = 0.20 \times 4.0 = \mathbf{0.80 \quad (80\%)}$.
* Current test suite composition:
  * 90% are fast unit tests: $S_1 = 1.0$ minute ($p_1 = 0.90$).
  * 10% are slow integration tests: $S_2 = 31.0$ minutes ($p_2 = 0.10$).
  * $\mathbb{E}[S] = 0.90(1) + 0.10(31) = 4.0$ minutes.
  * $\mathbb{E}[S^2] = 0.90(1^2) + 0.10(31^2) = 0.90 + 96.10 = 97.0 \text{ min}^2$.
  * $C_v^2 = \frac{97.0 - 16.0}{16.0} = 5.0625$.

### Baseline Queue Delay:
Using the P-K formula:

$$
W_q = \left(\frac{\rho \mathbb{E}[S]}{1 - \rho}\right) \left(\frac{1 + C_v^2}{2}\right) = \left(\frac{0.80 \times 4.0}{1 - 0.80}\right) \left(\frac{1 + 5.0625}{2}\right)
$$

$$
W_q = \left(\frac{3.2}{0.20}\right) \left(\frac{6.0625}{2}\right) = 16.0 \times 3.03125 = \mathbf{48.50 \text{ minutes}}
$$

Developers wait an average of **$48.5$ minutes** in queue for a test suite whose average execution time is only $4.0$ minutes! Total turnaround time is $W = 48.5 + 4.0 = 52.5$ minutes.

---

### Proposal A: The Hardware Purchasing Strategy (Scale Up Servers)
Management proposes upgrading the server instance to a 25% faster CPU, reducing all execution runtimes by 20% ($\mathbb{E}[S] \to 3.2$ minutes):
* New utilization: $\rho_{\text{new}} = 0.20 \times 3.2 = 0.64$ ($64\%$).
* Variance: $\mathbb{E}[S^2]_{\text{new}} = 0.90(0.8^2) + 0.10(24.8^2) = 0.576 + 61.504 = 62.08$.
* $C_v^2$ remains unchanged at $5.0625$.
* New wait time:
  $$W_{q, A} = \left(\frac{0.64 \times 3.2}{1 - 0.64}\right) \times 3.03125 = \left(\frac{2.048}{0.36}\right) \times 3.03125 = 5.6889 \times 3.03125 = \mathbf{17.24 \text{ minutes}}$$
* Latency reduction: From $48.5$ min to $17.2$ min (a **$64.5\%$ reduction**).

---

### Proposal B: The Software Engineering Strategy (Shard the Suite)
The software team leaves the hardware unchanged (keeping 1 server at $\mu = 0.25$, $\rho = 0.80$).
Instead, they spend 3 days refactoring the test harness:
* The 31-minute integration test is split into 31 parallel, uniform 1-minute test targets.
* Now, all tests in the system take approximately $S \approx 1.0$ minute.
* The test execution duration has negligible variance: $C_v^2 \approx 0.10$.
* Mean job duration remains $\mathbb{E}[S] = 4.0$ min of work (or handled via uniform sharding).
* New wait time:
  $$W_{q, B} = \left(\frac{0.80 \times 4.0}{1 - 0.80}\right) \times \left(\frac{1 + 0.10}{2}\right) = 16.0 \times 0.55 = \mathbf{8.80 \text{ minutes}}$$
* Latency reduction: From $48.5$ min to $8.80$ min (an **$81.9\%$ reduction**)!

### The Financial & Operational Verdict:

| Architecture Strategy | Hardware Budget | Fleet Utilization $\rho$ | Variance Factor $\frac{1 + C_v^2}{2}$ | Mean Queue Wait $W_q$ |
| :--- | :---: | :---: | :---: | :---: |
| **Baseline (High Variance)** | Baseline | 80% | 3.03 | **48.5 min** |
| **Proposal A (Buy Faster CPU)** | +25% Cost | 64% | 3.03 | **17.2 min** |
| **Proposal B (Shard Test Suite)** | **$0 Cost** | **80%** | **0.55** | **8.8 min** |

**The Lesson**: Reducing variance through test sharding achieved nearly **double the latency reduction of upgrading hardware**, with **zero ongoing cloud infrastructure expense**.

---

## 7. Hands-On 3-Tier Scaffolding Challenges

### 🟢 Warmup (Tier 1): Analytical Comparison of $M/D/1$ vs $M/M/1$
**Scenario**: A build server handles pull requests arriving at $\lambda = 8$ PRs/hour. The average test duration is $\mathbb{E}[S] = 6$ minutes ($0.10$ hours, $\mu = 10$ PRs/hour). Utilization is $\rho = 0.80$.

1. Calculate the mean queue wait time $W_q$ assuming service times are exponential ($M/M/1$, $C_v^2 = 1.0$).
2. Calculate the mean queue wait time $W_q$ assuming service times are deterministic ($M/D/1$, $C_v^2 = 0.0$).
3. By what percentage does eliminating service variance reduce developer queue waiting time?

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **$M/M/1$ Waiting Time**:
   $$W_{q, M/M/1} = \frac{\rho \mathbb{E}[S]}{1 - \rho} = \frac{0.80 \times 6.0}{1 - 0.80} = \frac{4.8}{0.20} = \mathbf{24.0 \text{ minutes}}$$

2. **$M/D/1$ Waiting Time**:
   $$W_{q, M/D/1} = \frac{\rho \mathbb{E}[S]}{1 - \rho} \cdot \left(\frac{1 + 0}{2}\right) = 24.0 \times 0.50 = \mathbf{12.0 \text{ minutes}}$$

3. **Percentage Reduction**:
   $$\frac{24.0 - 12.0}{24.0} \times 100\% = \mathbf{50.0\%}$$
   Eliminating service variability cuts waiting time in half.
</details>

---

### 🟡 Diagnostic (Tier 2): Evaluating the Impact of Outlier Test Targets
**Scenario**: A test suite has 99 fast tests that each take $1.0$ minute, and 1 monster test that takes $T$ minutes. Each test target is submitted with equal probability ($p_{\text{fast}} = 0.99, p_{\text{slow}} = 0.01$).
Arrival rate is fixed at $\lambda = 0.10$ jobs/min.
1. Write the expression for $\mathbb{E}[S]$, $\mathbb{E}[S^2]$, and $\rho$ as functions of $T$.
2. For $T = 1.0$ min (all uniform), calculate $\rho$ and $W_q$.
3. For $T = 31.0$ min, calculate $\rho$ and $W_q$.
4. For $T = 71.0$ min, calculate $\rho$ and $W_q$. Notice how the 1% outlier dominates total queue delay.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Formulations**:
   $$\mathbb{E}[S] = 0.99(1) + 0.01(T) = 0.99 + 0.01 T$$
   $$\mathbb{E}[S^2] = 0.99(1^2) + 0.01(T^2) = 0.99 + 0.01 T^2$$
   $$\rho = \lambda \mathbb{E}[S] = 0.10(0.99 + 0.01 T)$$
   $$W_q = \frac{\lambda \mathbb{E}[S^2]}{2(1 - \rho)} = \frac{0.10(0.99 + 0.01 T^2)}{2(1 - 0.10(0.99 + 0.01 T))}$$

2. **$T = 1.0$ min (Uniform)**:
   $$\mathbb{E}[S] = 1.0, \quad \mathbb{E}[S^2] = 1.0, \quad \rho = 0.10$$
   $$W_q = \frac{0.10(1.0)}{2(1 - 0.10)} = \frac{0.10}{1.80} \approx \mathbf{0.0556 \text{ minutes (3.3 seconds)}}$$

3. **$T = 31.0$ min**:
   $$\mathbb{E}[S] = 0.99 + 0.31 = 1.30 \text{ min}$$
   $$\mathbb{E}[S^2] = 0.99 + 0.01(961) = 0.99 + 9.61 = 10.60 \text{ min}^2$$
   $$\rho = 0.10 \times 1.30 = 0.13 \quad (13\% \text{ utilization})$$
   $$W_q = \frac{0.10 \times 10.60}{2(1 - 0.13)} = \frac{1.060}{1.74} \approx \mathbf{0.609 \text{ minutes (36.5 seconds)}}$$
   Waiting time increased by **$11\times$**, while utilization only went from $10\%$ to $13\%$.

4. **$T = 71.0$ min**:
   $$\mathbb{E}[S] = 0.99 + 0.71 = 1.70 \text{ min}$$
   $$\mathbb{E}[S^2] = 0.99 + 0.01(5041) = 0.99 + 50.41 = 51.40 \text{ min}^2$$
   $$\rho = 0.10 \times 1.70 = 0.17 \quad (17\% \text{ utilization})$$
   $$W_q = \frac{0.10 \times 51.40}{2(1 - 0.17)} = \frac{5.14}{1.66} \approx \mathbf{3.096 \text{ minutes}}$$
   Even at only **$17\%$ server utilization**, waiting time has exploded to over $3$ minutes—a **$55\times$ increase** driven purely by a 1% outlier!
</details>

---

### 🔴 Capstone Challenge (Tier 3): Truncated Pareto Timeouts to Tame Heavy Tails
**Scenario**: A cloud worker pool executes jobs whose durations follow an unbounded Pareto distribution with shape $\alpha = 1.5$ and scale $s_m = 1.0$ minute:
$$f(s) = \frac{1.5}{s^{2.5}}, \quad s \ge 1.0$$
Because $\alpha = 1.5 \le 2$, $\mathbb{E}[S^2] = \infty$, so unconstrained execution results in $W_q = \infty$ for any arrival rate $\lambda > 0$.

To stabilize the queue, you implement a hard watchdog timeout at $T_{\text{max}} = 60$ minutes. Any job that reaches 60 minutes is forcibly killed, yielding a **Truncated Pareto distribution**:
$$S_{\text{trunc}} = \min(S, T_{\text{max}})$$

1. Compute the probability $p_{\text{kill}} = \mathbb{P}(S > T_{\text{max}})$ that a job is killed.
2. Compute the first moment $\mathbb{E}[S_{\text{trunc}}]$ and second moment $\mathbb{E}[S_{\text{trunc}}^2]$ under the 60-minute truncation cap.
3. For arrival rate $\lambda = 0.20$ jobs/min, compute the stabilized utilization $\rho$ and mean waiting time $W_q$.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
1. **Kill Probability**:
   $$p_{\text{kill}} = \mathbb{P}(S > 60) = \left(\frac{1.0}{60}\right)^{1.5} = \frac{1}{60 \sqrt{60}} = \frac{1}{464.76} \approx \mathbf{0.00215 \quad (0.215\%)}$$
   Only roughly 1 in every 465 jobs is killed.

2. **Moments of Truncated Pareto**:
   The expectation of $\min(S, T)$ can be computed via integration of the survival function:
   $$\mathbb{E}[S_{\text{trunc}}] = \int_0^{T_{\max}} \mathbb{P}(S > s) \, ds = \int_0^1 1 \, ds + \int_1^{60} s^{-1.5} \, ds$$
   $$\mathbb{E}[S_{\text{trunc}}] = 1 + \left[ \frac{s^{-0.5}}{-0.5} \right]_1^{60} = 1 - 2\left(\frac{1}{\sqrt{60}} - 1\right) = 1 + 2\left(1 - \frac{1}{7.746}\right) = 1 + 2(0.8709) \approx \mathbf{2.742 \text{ minutes}}$$

   For the second moment:
   $$\mathbb{E}[S_{\text{trunc}}^2] = \int_1^{60} s^2 \left(\frac{1.5}{s^{2.5}}\right) ds + 60^2 \cdot \mathbb{P}(S > 60)$$
   $$\int_1^{60} 1.5 s^{-0.5} \, ds = 1.5 \left[ 2 \sqrt{s} \right]_1^{60} = 3.0(\sqrt{60} - 1) = 3.0(7.746 - 1) = 20.238$$
   $$60^2 \times 0.0021517 = 3600 \times 0.0021517 \approx 7.746$$
   $$\mathbb{E}[S_{\text{trunc}}^2] = 20.238 + 7.746 = \mathbf{27.984 \text{ min}^2}$$
   Truncation has rendered the infinite second moment finite!

3. **Queue Stabilization at $\lambda = 0.20$**:
   $$\rho = \lambda \mathbb{E}[S_{\text{trunc}}] = 0.20 \times 2.742 = \mathbf{0.5484 \quad (54.8\%)}$$
   Using the P-K formula:
   $$W_q = \frac{\lambda \mathbb{E}[S_{\text{trunc}}^2]}{2(1 - \rho)} = \frac{0.20 \times 27.984}{2(1 - 0.5484)} = \frac{5.5968}{2(0.4516)} = \frac{5.5968}{0.9032} \approx \mathbf{6.197 \text{ minutes}}$$

**Key Engineering Takeaway**: By killing the slowest $0.2\%$ of jobs with a 60-minute watchdog timeout, an infinite expected queue wait ($W_q = \infty$) is collapsed down to a predictable, manageable **$6.2$ minutes**!
</details>

---

**[Course Home](../README.md) | [Python Lab](../python/05_service_variance_and_pollaczek_khinchine.ipynb) | [Prev: Ch 04: Renewal Theory & Inspection Paradox](04_the_inspection_paradox_and_renewal_theory.ipynb) | [Next: Ch 06: Kingman Heavy-Traffic](../README.md)**
